Considere um MDP com três estados, (1, 2, 3) , com recompensas −1, −2, 0, respectivamente. O estado 3 é um estado terminal. Nos estados 1 e 2, há duas ações possíveis: a e b. O modelo de transição é como segue:

•  No estado 1, a ação a move o agente para o estado 2 com probabilidade 0,8 ou faz com que o agente permaneça no mesmo estado com probabilidade 0,2.

• No estado 2, a ação a move o agente para o estado 1 com probabilidade 0,8 ou faz com que o agente permaneça no mesmo estado com probabilidade 0,2.

• No estado 1 ou estado 2, a ação b move o agente para o estado 3 com probabilidade 0,1 ou faz com que o agente permaneça no mesmo estado com probabilidade 0,9.

In [ ]:
from copy import deepcopy
import numpy as np

from collections import defaultdict

In [ ]:
def createMDP(gamma=0.9):

    '''
    definição do ambiente
    '''

    S       = [i for i in range(1,4)]

    goals   = 3
    actions = ["a", "b",]
    A       = {s : actions
               for s in S }

    R        = {s : -1*s if s != goals else 0 for s in S }

    # Lista de transições (p,s')
    P        = {
        (1,'a'): [(0.8,2), (0.2,1)],
        (1,'b'): [(0.1,3), (0.9,1)],
        (2,'a'): [(0.8,1), (0.2,2)],
        (2,'b'): [(0.1,3), (0.9,2)],
        (3,None): [(1.0,3)],

            }

    gamma    = gamma

    return (S,A,R,P,gamma)



### Algoritmo Policy Iteration

In [ ]:
def policy_iteration(mdp, eval_theta=1e-9,pi0 ={1:'a', 2:'a', 3:None}):
    S, A, R, P, gamma = mdp
    # Política inicial 
    pi = pi0
    V = {s: 0.0 for s in S}
    V[3] = 0.0

    iter_count = 0
    while True:
        iter_count += 1
        print(f"\n===== Iteração {iter_count} =====")
        print(f"Política atual: {pi}")

        # 1) Avaliação da política
        delta_total = 0.0
        while True:
            delta = 0.0
            for s in S:
                if s == 3:
                    continue
                v_old = V[s]
                a = pi[s]
                V[s] = sum(prob * (R[s] + gamma * V[s2]) for prob, s2 in P[(s,a)])
                delta = max(delta, abs(v_old - V[s]))
            delta_total += delta
            if delta < eval_theta:
                break
        print(f"Etapa 1 concluída (Avaliação da política) — Δ total: {delta_total:.2e}")

        # Mostra valores atuais de V(s)
        print("Valores de estado após avaliação:")
        for s in S:
            print(f"  V({s}) = {V[s]:.6f}")

        # 2) Melhoria da política
        print("\n[Etapa 2] Melhoria da política:")
        policy_stable = True
        for s in S:
            if s == 3:
                continue
            old_a = pi[s]
            best_a = max(A[s],
                         key=lambda a: sum(prob * (R[s] + gamma * V[s2])
                                           for prob, s2 in P[(s,a)]))
            # imprime os Q-valores resumidos
            q_vals = {a: sum(prob * (R[s] + gamma * V[s2]) for prob, s2 in P[(s,a)]) for a in A[s]}
            print(f"  Estado {s}: Q(a)={q_vals} → melhor ação={best_a}")
            
            pi[s] = best_a
            if best_a != old_a:
                print(f"    Política muda: {old_a} → {best_a}")
                policy_stable = False

        if policy_stable:
            print("\nPolítica estabilizou — encerrando iterações")
            print("\n===== RESULTADO FINAL =====")
            for s in S:
                print(f"Estado {s}: melhor ação = {pi[s]}, valor = {V[s]:.6f}")
            return pi, V


### Ação `b` inicial em ambos os estados

In [ ]:
def main(pi0={1:'b', 2:'b', 3:None}):
    mdp = createMDP(gamma=0.9)

    print("\n== Policy Iteration ==")
    pi_PI, V_PI = policy_iteration(mdp,pi0=pi0)
    print("Política ótima (PI):", pi_PI)   
    print("Valores (PI):", V_PI)

main()

### Ação inicial `a` em ambos os estados

In [ ]:
main(pi0={1:'a', 2:'a', 3:None})